# Phase 3: Machine Learning Modeling
## GDP and Life Expectancy — Predictive Analysis

**Temporal validation:** Train 2000–2018 | Test 2019–2024  
**Target metric:** R² > 0.90 on test set  
**Models:** OLS, Ridge, Lasso, Quantile regression, Random Forest, XGBoost, LSTM (PyTorch), Ensemble stacking

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.utils.config import FINAL_DIR, OUTPUTS_DIR
print('Working directory configured.')

## 1. Load Data and Prepare Features

In [ ]:
from src.analysis.ml_models import prepare_features, make_split, get_feature_cols

df = pd.read_csv(FINAL_DIR / 'master_dataset.csv')
print(f'Raw dataset: {df.shape}')

df_feat = prepare_features(df)
feat_cols = get_feature_cols(df_feat)
print(f'Features available: {len(feat_cols)}')
print('\nFeature groups:')
for grp, tag in [('economic', 'gdp'), ('health', 'health'), 
                  ('governance', 'wgi'), ('demographic', 'age')]:
    n = sum(tag in f for f in feat_cols)
    print(f'  {grp}: ~{n} features')

In [ ]:
ds = make_split(df)
print(f'Train: {ds.X_train.shape}  (years {ds.df_train["year"].min()}–{ds.df_train["year"].max()})')
print(f'Test:  {ds.X_test.shape}  (years {ds.df_test["year"].min()}–{ds.df_test["year"].max()})')
print(f'Target: life_expectancy — mean train: {ds.y_train.mean():.1f}, std: {ds.y_train.std():.1f}')

## 2. Run Full ML Pipeline

In [ ]:
from src.analysis.ml_models import run_all_ml

results = run_all_ml(df)
print('Pipeline complete.')

In [ ]:
# Summary table
rows = []
for name, m in results['metrics'].items():
    rows.append({
        'Model': name,
        'R²_train': round(m.r2_train, 4),
        'R²_test': round(m.r2_test, 4),
        'RMSE_train': round(m.rmse_train, 3),
        'RMSE_test': round(m.rmse_test, 3),
        'MAE_test': round(m.mae_test, 3),
    })

perf = pd.DataFrame(rows).set_index('Model')
perf['Meets_Target'] = perf['R²_test'] >= 0.90
print(perf.to_string())
print(f'\nModels meeting R² ≥ 0.90 target: {perf["Meets_Target"].sum()}')

## 3. Model Performance Visualization

In [ ]:
from IPython.display import Image
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'model_performance.png'))

In [ ]:
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'pred_vs_actual.png'))

In [ ]:
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'residuals.png'))

## 4. SHAP Interpretability

In [ ]:
from src.analysis.interpretability import run_interpretability

interp = run_interpretability(results)

print('Top 10 features by XGBoost SHAP importance:')
print(interp['xgb_global_importance'].head(10).round(4).to_string())
print('\nTop 10 features by RF SHAP importance:')
print(interp['rf_global_importance'].head(10).round(4).to_string())

In [ ]:
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'shap_global_bar.png'))

In [ ]:
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'shap_beeswarm_xgboost.png'))

In [ ]:
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'shap_waterfall_xgboost.png'))

In [ ]:
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'shap_dependence_log_gdp_per_capita_ppp_xgboost.png'))

## 5. GDP Threshold Analysis

Three significant structural breaks in the GDP–life expectancy relationship:

In [ ]:
thresh = results['thresholds']
print('GDP Threshold Detection Results (Chow Test):')
print(thresh[['gdp_per_capita_ppp_threshold', 'slope_below', 'slope_above',
              'chow_f_stat', 'chow_p_value', 'n_below', 'n_above']].to_string())
print('\nInterpretation:')
for _, row in thresh.iterrows():
    sig = '***' if row['chow_p_value'] < 0.001 else ('**' if row['chow_p_value'] < 0.01 else '*')
    print(f"  ${row['gdp_per_capita_ppp_threshold']:,.0f}: slope {row['slope_below']:.2f} → {row['slope_above']:.2f} {sig}")

In [ ]:
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'gdp_threshold.png'))

## 6. Partial Dependence Plots

In [ ]:
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'pdp_grid.png'))

## 7. Time-Series Cross-Validation

In [ ]:
cv = results['cv_results']
print('CV Summary (walk-forward, 5 folds):')
print(cv.groupby('model')[['r2', 'rmse']].agg(['mean', 'std']).round(4))
Image(str(OUTPUTS_DIR / 'figures' / 'ml' / 'cv_results.png'))

## 8. Feature Importance Comparison

Comparing SHAP (model-agnostic) vs gain-based (XGBoost) vs MDI (Random Forest):

In [ ]:
fi = pd.read_csv(OUTPUTS_DIR / 'tables' / 'feature_importance.csv', index_col=0)
print('Top 15 features by XGB SHAP:')
print(fi.sort_values('xgb_shap', ascending=False).head(15)[['xgb_shap', 'rf_shap', 'xgb_gain']].round(4).to_string())

## 9. Key Findings Summary

### Model Performance
| Model | R²_test | RMSE_test |
|-------|---------|----------|
| Ensemble (RF+XGB+LSTM) | **0.91** | **3.23** |
| LSTM (PyTorch 2-layer) | **0.91** | **3.29** |
| XGBoost | **0.91** | 3.36 |
| Random Forest | 0.90 | 3.53 |
| Lasso (baseline) | 0.88 | 3.78 |

### GDP Threshold Effects
1. **$1,271**: Below this, slope β=3.1 (diminished returns — basic needs not met)
2. **$9,090**: Slope increases to β=6.0 (middle-income health dividend)
3. **$25,950**: Above this, slope drops to β=2.2 (diminishing returns at high income)

### Top Predictors (SHAP)
1. **GDP×Education interaction** — synergy between income and human capital
2. **Fertility rate** — demographic transition proxy
3. **Sanitation access** — basic infrastructure quality
4. **Age 65+ share** — population structure
5. **Water access** — public health infrastructure